<a href="https://colab.research.google.com/github/bahawal-khan/LANGCHAIN-Practice/blob/main/RAG%20pipeline/RAG_pipeline_using_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Step-1 Text Document creation**

In [2]:
text = """
Artificial Intelligence is a field of computer science that focuses on creating systems
capable of performing tasks that normally require human intelligence.

Machine Learning is a subset of Artificial Intelligence. It allows computers to learn
patterns from data and make predictions without being explicitly programmed.

Deep Learning is a subset of Machine Learning that uses artificial neural networks with
multiple layers. Deep learning is widely used in computer vision, natural language
processing, speech recognition, and many other applications.

Natural Language Processing enables computers to understand, process, and generate
human language. NLP is used in chatbots, machine translation, sentiment analysis,
text classification, and question answering.

Computer Vision enables computers to understand and analyze images and videos.
It is commonly used for image classification, object detection, face recognition,
and medical image analysis.
"""

with open('document.txt','w') as f:
  f.write(text)

In [3]:
!pip install -q langchain-community


**Step-2 Text Document Load**

In [4]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader('document.txt')
docs = loader.load()

print(docs[0].page_content)
print()
print(docs[0].metadata)
print()
print(len(docs))

/tmp/ipykernel_18108/2648352147.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader



Artificial Intelligence is a field of computer science that focuses on creating systems
capable of performing tasks that normally require human intelligence.

Machine Learning is a subset of Artificial Intelligence. It allows computers to learn
patterns from data and make predictions without being explicitly programmed.

Deep Learning is a subset of Machine Learning that uses artificial neural networks with
multiple layers. Deep learning is widely used in computer vision, natural language
processing, speech recognition, and many other applications.

Natural Language Processing enables computers to understand, process, and generate
human language. NLP is used in chatbots, machine translation, sentiment analysis,
text classification, and question answering.

Computer Vision enables computers to understand and analyze images and videos.
It is commonly used for image classification, object detection, face recognition,
and medical image analysis.


{'source': 'document.txt'}

1


In [5]:
!pip install -q langchain-text-splitters

**Step-3 We Do Chunking**

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter =  RecursiveCharacterTextSplitter(
    chunk_size = 300,
    chunk_overlap = 50
)


chunks = text_splitter.split_documents(docs)
print(len(chunks))
print()
print(chunks[0].page_content)
print()
print(chunks[0].metadata)

5

Artificial Intelligence is a field of computer science that focuses on creating systems
capable of performing tasks that normally require human intelligence.

{'source': 'document.txt'}


In [7]:
!pip install -q langchain-huggingface sentence-transformers

**Step-4 Now Make Embedding Of These Chunks using chroma and store into vector store**

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings  = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
pip install chromadb langchain-chroma

In [10]:
!pip install -U opentelemetry-api opentelemetry-sdk

In [11]:
!pip install -U --force-reinstall \
    opentelemetry-api \
    opentelemetry-sdk \
    opentelemetry-semantic-conventions

  Using cached opentelemetry_api-1.44.0-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_sdk-1.44.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached opentelemetry_semantic_conventions-0.65b0-py3-none-any.whl.metadata (2.4 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
Using cached opentelemetry_api-1.44.0-py3-none-any.whl (60 kB)
Using cached opentelemetry_sdk-1.44.0-py3-none-any.whl (137 kB)
Using cached opentelemetry_semantic_conventions-0.65b0-py3-none-any.whl (204 kB)
Using cached typing_extensions-4.16.0-py3-none-any.whl (45 kB)
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.16.0
    Uninstalling typing_extensions-4.16.0:
      Successfully uninstalled typing_extensions-4.16.0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.44.0
    Uninstalling opentelemetry-api-1.44.0:
      Successfully uninstalled opentelemetry-api-1.44.0
  Attempting u

In [12]:
from langchain_chroma import Chroma
vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding = embeddings,
    collection_name = 'rg_collection'
)

**Step-5 User Query and make vector of user query then do a sementic search between user query and already stored vectors**

In [36]:
query = input('Ask Your Question: ')

retriever = vectorstore.as_retriever(
    search_kwargs = {'k': 1}
)

relevent_docs =  retriever.invoke(query)

for docs in relevent_docs:
  print(docs.page_content)

Ask Your Question: What is NLP?
Natural Language Processing enables computers to understand, process, and generate
human language. NLP is used in chatbots, machine translation, sentiment analysis,
text classification, and question answering.


**Step-6 Now combine the chunks we will give the chunks + user query to LLM.**

In [37]:
context = '\n\n'.join(
    doc.page_content for doc in relevent_docs
)
print(context)


Natural Language Processing enables computers to understand, process, and generate
human language. NLP is used in chatbots, machine translation, sentiment analysis,
text classification, and question answering.


In [38]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    (
        'system',
         "Answer the user's question using only the provided context. "
        "If the answer is not available in the context, say that you don't know."
    ),

    (
        'human',

        """
        Context: {context}

        Question: {query}
"""
    )
])


prompt_value = prompt.invoke({
    'context': context,
    'query': query
})

print(prompt_value)

messages=[SystemMessage(content="Answer the user's question using only the provided context. If the answer is not available in the context, say that you don't know.", additional_kwargs={}, response_metadata={}), HumanMessage(content='\n        Context: Natural Language Processing enables computers to understand, process, and generate\nhuman language. NLP is used in chatbots, machine translation, sentiment analysis,\ntext classification, and question answering.\n\n        Question: What is NLP?\n', additional_kwargs={}, response_metadata={})]


**step-7 Create a chatmodel and generate a response from him.**

In [25]:
pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.1 MB/s eta 0:00:00


In [29]:
from langchain_groq import ChatGroq
from google.colab import userdata

groq_api_key = userdata.get("GROQ_API_KEY")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    api_key=groq_api_key
)

In [39]:
response = llm.invoke(prompt_value)
print(response.content)

Natural Language Processing (NLP) is a field of computer science that enables computers to understand, process, and generate human language. It is used in applications such as chatbots, machine translation, sentiment analysis, text classification, and question answering.
